# Airline Satisfaction — XGBoost experiment

This separate notebook trains XGBoost for `satisfaction`, evaluates it with 5-fold stratified cross-validation, and creates a test submission by averaging the five fold models. It leaves the CatBoost notebook unchanged.


## 1. Install and import packages

Run the install command if XGBoost is not already installed in this notebook's Python environment.


In [1]:
%pip install xgboost


   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 1.2 MB/s eta 0:00:40
    --------------------------------------- 1.0/48.9 MB 1.6 MB/s eta 0:00:30
   - -------------------------------------- 1.3/48.9 MB 1.6 MB/s eta 0:00:31
   - -------------------------------------- 1.6/48.9 MB 1.6 MB/s eta 0:00:29
   - -------------------------------------- 2.1/48.9 MB 1.7 MB/s eta 0:00:29
   -- ------------------------------------- 2.6/48.9 MB 1.8 MB/s eta 0:00:27
   -- ------------------------------------- 2.6/48.9 MB 1.8 MB/s eta 0:00:27
   -- ------------------------------------- 3.4/48.9 MB 1.8 MB/s eta 0:00:25
   -- ------------------------------------- 3.7/48.9 MB 1.7 MB/s eta 0:00:27
   --- ------------------------------------ 3.9/48.9 MB 1.8 MB/s eta 0:00:25
   --- ------------------------------------ 4.5/48.9 MB 1.8 MB/s eta 0:00:25
   --- ------

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


## 2. Load the competition data

`train (1).csv` contains the features and known satisfaction label. `test.csv` has no labels and is only used to create submission predictions.


In [3]:
DATA_DIR = Path.cwd()
train = pd.read_csv(DATA_DIR / "train (1).csv")
test = pd.read_csv(DATA_DIR / "test.csv")
TARGET = "satisfaction"
ID_COL = "id"
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head())


Train shape: (699635, 23)
Test shape: (299844, 22)
   id  Age  Flight Distance  Inflight wifi service  \
0   0   36              694                      4   
1   1   56              651                      5   
2   2   31             1371                      3   
3   3   10             1616                      3   
4   4   23              481                      3   

   Departure/Arrival time convenient  Ease of Online booking  Gate location  \
0                                  4                       4              3   
1                                  5                       5              5   
2                                  4                       2              5   
3                                  5                       3              3   
4                                  4                       3              4   

   Food and drink  Online boarding  Seat comfort  ...  Baggage handling  \
0               1                4             1  ...                 4   

## 3. Prepare features and encode categories

XGBoost needs numeric input here, so one-hot encoding turns each category into indicator columns. `Travel_Context` combines travel type, customer type, and class, matching the CatBoost feature setup. The encoder is fitted separately inside each fold using only that fold's training rows.


In [4]:
X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

context_cols = ["Type of Travel", "Customer Type", "Class"]
X["Travel_Context"] = X[context_cols].astype(str).agg(" | ".join, axis=1)
X_test["Travel_Context"] = X_test[context_cols].astype(str).agg(" | ".join, axis=1)

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [col for col in X.columns if col not in cat_cols]
for col in cat_cols:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)
for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print("Categorical columns:", cat_cols)
print("Numeric columns:", len(num_cols))
print("Satisfied share:", y.mean())


Categorical columns: ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'Travel_Context']
Numeric columns: 17
Satisfied share: 0.44357272006117476


## 4. Run 5-fold CV and make an ensemble submission

The labeled data is split into five folds. Each model trains on four folds and validates on the fifth; every row is held out once. We record each fold's ROC AUC and the mean, and save out-of-fold predictions. Each fold model also predicts `test.csv`; averaging those five probability predictions creates the submission.

This is five full model fits, so it will take longer than one training run. These XGBoost settings are a starting point, not a guarantee of beating CatBoost.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_proba = np.zeros(len(X), dtype=float)
test_proba_sum = np.zeros(len(X_test), dtype=float)
fold_aucs = []
fold_best_trees = []

for fold_number, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    print(f"\nFold {fold_number}/5")
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
            ("numeric", "passthrough", num_cols),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )
    X_fit = preprocessor.fit_transform(X.iloc[fit_idx])
    X_valid = preprocessor.transform(X.iloc[valid_idx])
    X_test_fold = preprocessor.transform(X_test)

    model = XGBClassifier(
        n_estimators=4000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        early_stopping_rounds=150,
        n_jobs=-1,
        random_state=42 + fold_number,
    )
    model.fit(
        X_fit, y.iloc[fit_idx],
        eval_set=[(X_valid, y.iloc[valid_idx])],
        verbose=300,
    )

    valid_proba = model.predict_proba(X_valid)[:, 1]
    fold_auc = roc_auc_score(y.iloc[valid_idx], valid_proba)
    oof_proba[valid_idx] = valid_proba
    test_proba_sum += model.predict_proba(X_test_fold)[:, 1]
    best_trees = model.best_iteration + 1 if model.best_iteration is not None else model.n_estimators
    fold_aucs.append(fold_auc)
    fold_best_trees.append(best_trees)
    print(f"Fold {fold_number} ROC AUC: {fold_auc:.6f}; best trees: {best_trees}")

mean_cv_auc = float(np.mean(fold_aucs))
std_cv_auc = float(np.std(fold_aucs, ddof=1))
oof_auc = roc_auc_score(y, oof_proba)
print("\nFold AUCs:", [round(score, 6) for score in fold_aucs])
print(f"Mean 5-fold ROC AUC: {mean_cv_auc:.6f} +/- {std_cv_auc:.6f}")
print(f"Combined OOF ROC AUC: {oof_auc:.6f}")

cv_results = pd.DataFrame({"fold": np.arange(1, 6), "roc_auc": fold_aucs, "best_trees": fold_best_trees})
cv_results.loc[len(cv_results)] = ["mean", mean_cv_auc, np.mean(fold_best_trees)]
cv_results_path = DATA_DIR / "xgboost_cv_results.csv"
cv_results.to_csv(cv_results_path, index=False)
print("Saved CV scores:", cv_results_path.resolve())

test_proba = test_proba_sum / 5
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_proba})
submission_path = DATA_DIR / "submission_xgboost_cv.csv"
submission.to_csv(submission_path, index=False)
print("Saved submission:", submission_path.resolve())
print(submission.head())
print("Rows:", len(submission), "IDs unique:", submission[ID_COL].is_unique)
print("Probabilities in [0, 1]:", submission[TARGET].between(0, 1).all())



Fold 1/5
[0]	validation_0-auc:0.94831
[300]	validation_0-auc:0.95819
[600]	validation_0-auc:0.95885


## Compare with CatBoost

Compare this mean 5-fold ROC AUC with CatBoost's mean 5-fold AUC. If scores are close, their predictions may still complement each other; only keep a blend if blended out-of-fold predictions improve ROC AUC. The output submission is `submission_xgboost_cv.csv`.
